# YOLOv8m Fine-Tuning — Multi-Source v3

**Mixed dataset (heterogeneous sources):**
- Roboflow export: 2919 frame (HD camera + drone footage)
- Auto-labeled: 5820 frame (İBB CCTV recordings)
- **Total: 8739 frames** across multiple cameras and quality levels

This multi-source approach enables cross-domain robustness — the model learns to detect vehicles across both high-quality and CCTV-grade footage.

**Önce:** Runtime → Change runtime type → **T4 GPU** seç!

In [ ]:
# 1. GPU kontrol
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    raise RuntimeError('GPU yok! Runtime → Change runtime type → T4 GPU')

In [ ]:
# 2. Drive mount (modeli direkt Drive'a kaydetmek için)
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os
SAVE_DIR = '/content/drive/MyDrive/tez_models/mobese_v3'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Save dir: {SAVE_DIR}')

In [ ]:
# 3. Ultralytics yükle
!pip install ultralytics -q

## Dataset Upload

Final dataset Drive'da: `/content/drive/MyDrive/datasets1/final_v3.zip`

Şu komut Drive'daki zip'i extract eder.

In [ ]:
# 4. Dataset extract
import zipfile
DATASET_ZIP = '/content/drive/MyDrive/Palmora_ML/datasets/datasets1/final_v3.zip'
DATASET_DIR = '/content/final_v3'

if os.path.exists(DATASET_ZIP):
    with zipfile.ZipFile(DATASET_ZIP, 'r') as z:
        z.extractall('/content/')
    print(f'Dataset extracted: {DATASET_DIR}')
else:
    print(f'HATA: {DATASET_ZIP} bulunamadı.')
    # Bulamazsa Drive'da nereye yüklendiğini ara
    import subprocess
    result = subprocess.run(
        ['find', '/content/drive/MyDrive', '-iname', 'final_v3.zip'],
        capture_output=True, text=True, timeout=120
    )
    print("\nDrive'da bulunan final_v3.zip dosyaları:")
    print(result.stdout if result.stdout else "Hiç bulunamadı")

In [ ]:
# 5. data.yaml düzenle (Colab path'leri için)
DATA_YAML = f'{DATASET_DIR}/data.yaml'

yaml_content = f'''train: {DATASET_DIR}/train/images
val:   {DATASET_DIR}/valid/images
test:  {DATASET_DIR}/test/images

nc: 4
names: ['bus', 'car', 'motorcycle', 'truck']
'''
with open(DATA_YAML, 'w') as f:
    f.write(yaml_content)

print('data.yaml updated')
!cat $DATA_YAML

In [ ]:
# 6. Dataset kontrol
import os
for split in ['train', 'valid', 'test']:
    img_count = len(os.listdir(f'{DATASET_DIR}/{split}/images'))
    lbl_count = len(os.listdir(f'{DATASET_DIR}/{split}/labels'))
    print(f'{split}: {img_count} images, {lbl_count} labels')

## Eğitim — YOLOv8m

In [ ]:
# 7. YOLOv8m fine-tuning
from ultralytics import YOLO

model = YOLO('yolov8m.pt')  # COCO pretrained

results = model.train(
    data=DATA_YAML,
    epochs=100,
    imgsz=640,
    batch=16,
    patience=25,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=5,
    cos_lr=True,
    label_smoothing=0.1,
    close_mosaic=10,
    augment=True,
    mosaic=1.0,
    mixup=0.15,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=5.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.0,    # Taralı alan yön-spesifik, flip kapalı
    flipud=0.0,
    name='mobese_v3',
    project=SAVE_DIR,
    save=True,
    plots=True,
    verbose=True,
)

print(f'\n✓ Eğitim bitti.')
print(f'Model: {SAVE_DIR}/mobese_v3/weights/best.pt')

In [ ]:
# 8. Test set üzerinde değerlendirme
metrics = model.val(data=DATA_YAML, split='test')
print(f'\n=== TEST SET RESULTS ===')
print(f'mAP50:    {metrics.box.map50:.4f}')
print(f'mAP50-95: {metrics.box.map:.4f}')
print(f'Precision: {metrics.box.mp:.4f}')
print(f'Recall:    {metrics.box.mr:.4f}')

for i, name in enumerate(['bus', 'car', 'motorcycle', 'truck']):
    print(f'  {name}: AP50={metrics.box.ap50[i]:.4f}')

In [ ]:
# 9. best.pt'yi indir
from google.colab import files
BEST_PT = f'{SAVE_DIR}/mobese_v3/weights/best.pt'
files.download(BEST_PT)